In [ ]:
!pip install fastapi uvicorn pyngrok nest_asyncio sentence-transformers faiss-cpu transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import faiss
import time
import nest_asyncio
from pyngrok import ngrok, conf
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- 1. قاعدة البيانات (تقدر تبدلها بملف الـ Excel بتاعك لاحقاً) ---
data = {
    "question": [
        "how can I track my order?",
        "what is your refund policy?",
        "how to contact support?",
        "hello", "hi"
    ],
    "answer": [
        "You can track your order using the tracking number sent to your email.",
        "Our refund policy allows returns within 30 days of purchase.",
        "You can contact support via email at support@example.com.",
        "Hello! How can I help you today?",
        "Hi there! Welcome to our customer support."
    ]
}
df = pd.DataFrame(data)

# --- 2. تعريف الـ FastAPI ---
app = FastAPI()

class QueryRequest(BaseModel):
    question: str

# --- 3. تجهيز الموديلات (RAG) ---
# تأكد إنك مفعل الـ T4 GPU من Runtime عشان السرعة
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ جاري تجهيز الموديلات على {DEVICE}... ثواني وهيكون الرابط جاهز.")

# موديل البحث (Embeddings)
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
corpus_embeddings = embedding_model.encode(df['question'].tolist(), convert_to_numpy=True)
faiss_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
faiss_index.add(corpus_embeddings.astype('float32'))

# موديل الرد (LLM) - نسخة خفيفة وسريعة
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

def get_answer(user_query):
    # البحث عن أقرب إجابة في البيانات
    q_emb = embedding_model.encode([user_query], convert_to_numpy=True)
    _, I = faiss_index.search(q_emb.astype('float32'), k=1)
    context = df.iloc[I[0][0]]['answer']

    # صياغة الرد النهائي
    prompt = f"<|im_start|>system\nYou are a helpful assistant. Context: {context}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = llm_model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1].strip()

# --- 4. المسارات (Endpoints) ---
@app.get("/")
def home():
    return {"status": "Online", "message": "Server is Running Successfully"}

@app.post("/ask")
def ask(request: QueryRequest):
    return {"answer": get_answer(request.question)}

# --- 5. تشغيل Ngrok والسيرفر (الحل الصافي) ---

# قفل أي عمليات قديمة إجبارياً
!pkill -9 ngrok
ngrok.kill()

# ضع الكود (Token) الخاص بك هنا
NGROK_AUTH_TOKEN = "3Dek8a7sAL5ctGroLeakYXMtmDe_7ygTCzDq8ZA9TieMWmadA"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# تغيير المنطقة لضمان عدم التعليق
conf.get_default().region = "eu"

try:
    # فتح النفق برابط عشوائي جديد
    public_url = ngrok.connect(8000, bind_tls=True)
    print("\n" + "="*60)
    print(f"🚀 تم تشغيل السيرفر بنجاح يا باسل!")
    print(f"🔗 الرابط الجديد: {public_url}")
    print(f"🛠️ واجهة الاختبار: {public_url}/docs")
    print("="*60 + "\n")
except Exception as e:
    print(f"❌ خطأ في تشغيل Ngrok: {e}")

# تشغيل الـ Event Loop لـ Colab
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# البدء!
await server.serve()

⏳ جاري تجهيز الموديلات على cpu... ثواني وهيكون الرابط جاهز.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


🚀 تم تشغيل السيرفر بنجاح يا باسل!
🔗 الرابط الجديد: NgrokTunnel: "https://exception-relight-consoling.ngrok-free.dev" -> "http://localhost:8000"
🛠️ واجهة الاختبار: NgrokTunnel: "https://exception-relight-consoling.ngrok-free.dev" -> "http://localhost:8000"/docs



INFO:     Started server process [5820]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5820]


In [ ]:
import pandas as pd
import torch
import faiss
import time
import nest_asyncio
from pyngrok import ngrok, conf
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- 1. سحب البيانات من رابط على النت ---
# استبدل هذا الرابط برابط ملف الـ CSV أو Excel المباشر الخاص بك
data_url = "https://raw.githubusercontent.com/bassel-wael/test-data/main/faq.csv"

try:
    print("🌐 جاري سحب البيانات من النت...")
    # لو الملف Excel استخدم pd.read_excel ولو CSV استخدم pd.read_csv
    if data_url.endswith('xlsx') or data_url.endswith('xls'):
        df = pd.read_excel(data_url)
    else:
        df = pd.read_csv(data_url)

    # تأكد إن الأعمدة في ملفك هي "question" و "answer" أو عدلها هنا
    df.columns = ['question', 'answer']
    print("✅ تم تحميل البيانات بنجاح.")
except Exception as e:
    print(f"⚠️ فشل سحب البيانات: {e}")
    print("استخدام بيانات افتراضية مؤقتاً...")
    df = pd.DataFrame({"question": ["hello"], "answer": ["Hi, I am your web-connected AI!"]})

# --- 2. إعداد السيرفر والموديلات ---
app = FastAPI()
class QueryRequest(BaseModel):
    question: str

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ جاري تجهيز الموديلات على {DEVICE}... (قد يستغرق وقتاً لو على CPU)")

# تحميل موديل الـ Embeddings للبحث
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
corpus_embeddings = embedding_model.encode(df['question'].tolist(), convert_to_numpy=True)
faiss_index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
faiss_index.add(corpus_embeddings.astype('float32'))

# تحميل موديل الرد (LLM)
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

def get_answer(user_query):
    # البحث عن السياق
    q_emb = embedding_model.encode([user_query], convert_to_numpy=True)
    _, I = faiss_index.search(q_emb.astype('float32'), k=1)
    context = df.iloc[I[0][0]]['answer']

    # صياغة الرد
    prompt = f"<|im_start|>system\nContext: {context}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = llm_model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1].strip()

@app.post("/ask")
def ask(request: QueryRequest):
    return {"answer": get_answer(request.question)}

@app.get("/")
def health():
    return {"status": "running"}

# --- 3. تشغيل Ngrok والسيرفر (الحل المتوافق مع كولاب) ---
# تنظيف العمليات القديمة
!pkill -9 ngrok
ngrok.kill()

# إعداد التوكن والمنطقة
NGROK_AUTH_TOKEN = "3Dek8a7sAL5ctGroLeakYXMtmDe_7ygTCzDq8ZA9TieMWmadA"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
conf.get_default().region = "eu"

try:
    public_url = ngrok.connect(8000, bind_tls=True)
    print("\n" + "="*60)
    print(f"🚀 السيرفر جاهز الآن!")
    print(f"🔗 الرابط العام: {public_url}")
    print(f"📂 واجهة التجربة: {public_url}/docs")
    print("="*60 + "\n")
except Exception as e:
    print(f"❌ خطأ Ngrok: {e}")

# التشغيل النهائي المتوافق مع Event Loop الخاص بكولاب
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

🌐 جاري سحب البيانات من النت...
⚠️ فشل سحب البيانات: HTTP Error 404: Not Found
استخدام بيانات افتراضية مؤقتاً...
⏳ جاري تجهيز الموديلات على cpu... (قد يستغرق وقتاً لو على CPU)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


🚀 السيرفر جاهز الآن!
🔗 الرابط العام: NgrokTunnel: "https://exception-relight-consoling.ngrok-free.dev" -> "http://localhost:8000"
📂 واجهة التجربة: NgrokTunnel: "https://exception-relight-consoling.ngrok-free.dev" -> "http://localhost:8000"/docs



INFO:     Started server process [5820]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [5820]


In [ ]:
!pip install faiss-cpu PyPDF2 python-docx pandas openpyxl sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.5 MB/s eta 0:00:00


In [ ]:
# --- 1. تثبيت كافة المكتبات اللازمة (عشان مطلعش ModuleNotFoundError) ---
!pip install faiss-cpu pyngrok uvicorn fastapi sentence-transformers PyPDF2 python-docx pandas openpyxl

import pandas as pd
import torch
import faiss
import nest_asyncio
from pyngrok import ngrok, conf
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import PyPDF2
from docx import Document
import os

# --- 2. وظائف قراءة المحتوى من الملفات المرفوعة ---

def read_pdf(file_path):
    text = ""
    if os.path.exists(file_path):
        with open(file_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                content = page.extract_text()
                if content: text += content + "\n"
    return text

def read_docx(file_path):
    if os.path.exists(file_path):
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs if para.text])
    return ""

def read_excel(file_path):
    if os.path.exists(file_path):
        df = pd.read_excel(file_path)
        return "\n".join(df.astype(str).apply(lambda x: " | ".join(x), axis=1).tolist())
    return ""

# --- 3. تجهيز البيانات وبناء الفهرس الذكي (RAG) ---

# المسارات اللي إنت حددتها
file_paths = [
    "/content/productsandprices.pdf",
    "/content/refundpolicy.docx",
    "/content/zbranches (1).xlsx"
]

all_chunks = []
print("⏳ جاري قراءة الملفات وتحويلها لقطع نصية...")

for path in file_paths:
    content = ""
    if path.endswith(".pdf"): content = read_pdf(path)
    elif path.endswith(".docx"): content = read_docx(path)
    elif path.endswith(".xlsx"): content = read_excel(path)

    if content:
        # تقسيم النص لقطع (Chunks) لزيادة دقة البحث
        temp_chunks = [content[i:i+600] for i in range(0, len(content), 600)]
        all_chunks.extend(temp_chunks)
        print(f"✅ تم معالجة: {path}")
    else:
        print(f"⚠️ تحذير: الملف غير موجود أو فارغ: {path}")

# إعداد الموديلات
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🤖 جاري تحميل الموديلات على {DEVICE} (قد يستغرق دقائق)...")

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
embeddings = embedding_model.encode(all_chunks, convert_to_numpy=True)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype('float32'))

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

# --- 4. إعداد الـ API (FastAPI) ---

app = FastAPI()

class Query(BaseModel):
    question: str

def generate_answer(user_query):
    # البحث عن أنسب معلومات في الملفات
    q_emb = embedding_model.encode([user_query], convert_to_numpy=True)
    D, I = index.search(q_emb.astype('float32'), k=3)
    context = "\n".join([all_chunks[idx] for idx in I[0]])

    prompt = f"""<|im_start|>system
أنت مساعد ذكي لمشروع DEPI. أجب على المستخدم باللغة العربية بناءً على السياق المرفق فقط.
السياق:
{context}
<|im_end|>
<|im_start|>user
{user_query}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = llm_model.generate(**inputs, max_new_tokens=250)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1].strip()

@app.post("/ask")
async def ask(request: Query):
    return {"answer": generate_answer(request.question)}

# --- 5. تشغيل نفق الـ Ngrok والسيرفر ---

!pkill -9 ngrok
# التوكن الخاص بك يا باسل
NGROK_TOKEN = "3Dek8a7sAL5ctGroLeakYXMtmDe_7ygTCzDq8ZA9TieMWmadA"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000, bind_tls=True)
print("\n" + "="*60)
print(f"🚀 السيرفر شغال بنجاح على الملفات الثلاثة!")
print(f"🔗 رابط الـ API العام: {public_url}")
print(f"📑 واجهة التجربة: {public_url}/docs")
print("="*60 + "\n")

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

⏳ جاري قراءة الملفات وتحويلها لقطع نصية...
✅ تم معالجة: /content/productsandprices.pdf
✅ تم معالجة: /content/refundpolicy.docx
✅ تم معالجة: /content/zbranches (1).xlsx
🤖 جاري تحميل الموديلات على cpu (قد يستغرق دقائق)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

INFO:     Started server process [1196]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🚀 السيرفر شغال بنجاح على الملفات الثلاثة!
🔗 رابط الـ API العام: NgrokTunnel: "https://exception-relight-consoling.ngrok-free.dev" -> "http://localhost:8000"
📑 واجهة التجربة: NgrokTunnel: "https://exception-relight-consoling.ngrok-free.dev" -> "http://localhost:8000"/docs

INFO:     197.46.185.93:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     197.46.185.93:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     197.46.185.93:0 - "POST /ask HTTP/1.1" 200 OK
